# Street Segment Safety Classification â€” Any City

End-to-end inference pipeline:
1. Set your target place below
2. Fetch OSM street network + features
3. Normalize features
4. Apply trained model (from `05_ClassificationML.ipynb`)
5. Export CSV + map

**Model was trained on London data.** The scaler and classifier are loaded from the `models/` folder.

## âš™ï¸ Configuration â€” Edit This Cell

In [ ]:
# ── EDIT HERE ────────────────────────────────────────────────────────────────
PLACE = "Puente de Vallecas, Madrid, Spain"   # Any OSM place string

#Raval, Barcelona, Spain
#Eixample, Barcelona, Spain
#Poble Sec, Barcelona, Spain

#Casco Viejo, Bilbao, Spain

#Triana, Seville, Spain

#Puente de Vallecas, Madrid, Spain
#Lavapiés, Madrid, Spain
#Arganzuela, Madrid, Spain
#Chamberí, Madrid, Spain
#Salamanca, Madrid, Spain


#Lagunillas, Málaga, Spain
# ─────────────────────────────────────────────────────────────────────────────

import os
import sys
from pathlib import Path

BASE_DIR   = r'C:\Users\maria\Documents\GitHub\MaCAD26-G01-DataEncoding - Barcelona\MaCAD26-G01-DataEncoding-BarcelonaVersion'
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
OUTPUT_DIR = os.path.join(BASE_DIR, 'csv')

# Make the repository root importable so `from scripts...` works regardless of launch directory.
ROOT_DIR = Path(BASE_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Prefer the local Geofabrik extract so building-derived features are available.
PBF_PATH = Path(BASE_DIR) / 'geofabrik' / 'england-260610.osm.pbf'
if not PBF_PATH.exists():
    PBF_PATH = None
else:
    PBF_PATH = str(PBF_PATH)

print(f'Target place : {PLACE}')
print(f'Model dir    : {MODEL_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')

: 

## Step 1: Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import joblib
from pathlib import Path
from scipy.spatial import cKDTree

# OSM timeout â€” generous to handle large queries
ox.settings.timeout = 300
ox.settings.log_console = False

CRS_METRIC = 'EPSG:3857'   # metres (UK= EPSG:27700); change to EPSG:3857 for non-UK cities
BUFFER_M   = 25             # segment buffer: 25m each side, flat ends
BUFFER_VIS = 50             # visibility / land-use buffer
RISK_COLORS = {'low': '#2ecc71', 'medium': '#f39c12', 'high': '#e74c3c'}
RISK_ORDER  = ['low', 'medium', 'high']

print('Imports OK')

## Step 2: Load Saved Model & Scaler

In [ ]:
scaler = joblib.load(os.path.join(MODEL_DIR, 'scaler.pkl'))
model  = joblib.load(os.path.join(MODEL_DIR, 'logistic_regression.pkl'))

print('âœ“ Scaler loaded')
print('âœ“ Model loaded:', type(model).__name__)
print('  Classes:', model.classes_)

## Step 3: Fetch Street Network

In [ ]:
print(f'Fetching street network for: {PLACE}')
place_parts = [part.strip() for part in PLACE.split(',') if part.strip()]
place_candidates = [', '.join(place_parts[i:]) for i in range(len(place_parts))]
place_candidates.append(PLACE)

def _repair_polygon(geom):
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type not in ('Polygon', 'MultiPolygon'):
        geom = geom.envelope
    try:
        if not geom.is_valid:
            repaired = geom.buffer(0)
            if repaired is not None and not repaired.is_empty:
                geom = repaired
    except Exception:
        pass
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type not in ('Polygon', 'MultiPolygon'):
        geom = geom.envelope
    return geom

study_area_ll = None
selected_query = None

for query in place_candidates:
    try:
        candidate = ox.geocode_to_gdf(query)
    except Exception as exc:
        print(f'  âš  Geocoding failed for {query!r}: {exc}')
        continue

    geom = _repair_polygon(candidate.geometry.iloc[0])
    if geom is not None and geom.geom_type in ('Polygon', 'MultiPolygon') and not geom.is_empty:
        study_area_ll = candidate.copy()
        study_area_ll.iloc[0, study_area_ll.columns.get_loc('geometry')] = geom
        selected_query = query
        break

if study_area_ll is None:
    raise TypeError(
        f"Could not resolve a usable polygon boundary for {PLACE!r}. "
        "Try a broader place name such as the containing city or borough."
    )

if selected_query != PLACE:
    print(f"  Using polygon boundary from fallback query: {selected_query!r}")

FEATURE_QUERY = selected_query or PLACE
print(f'  Using feature query: {FEATURE_QUERY!r}')

boundary = study_area_ll.geometry.iloc[0]
G = ox.graph_from_polygon(boundary, network_type='drive')
edges = ox.graph_to_gdfs(G, nodes=False).to_crs(CRS_METRIC)
nodes_gdf = ox.graph_to_gdfs(G, edges=False).to_crs(CRS_METRIC)
study_area = study_area_ll.to_crs(CRS_METRIC)
boundary = study_area.geometry.iloc[0]

edges = edges.reset_index()
print(f'âœ“ {len(edges):,} street segments fetched')

## Step 4: Fetch Building Footprints

Default: OSM (Overpass) first, then local PBF fallback.
UK/England places: local England PBF first, then OSM fallback if needed.

In [ ]:
def _repair_geometries(geoms):
    repaired = []
    for geom in geoms:
        if geom is None or geom.is_empty:
            repaired.append(geom)
            continue
        if not geom.is_valid:
            try:
                geom = geom.buffer(0)
            except Exception:
                pass
        repaired.append(geom)
    return gpd.GeoSeries(repaired, index=geoms.index, crs=geoms.crs)


def _is_uk_or_england_query(place: str) -> bool:
    p = place.strip().lower()
    return (
        p.endswith(', uk')
        or p.endswith(', united kingdom')
        or p.endswith(', england')
        or ', uk' in p
        or ', united kingdom' in p
        or ', england' in p
        or 'london borough of' in p
        or ('borough' in p and ', london, uk' in p)
    )


def _load_from_osm(place, crs):
    print('  Trying OSM (Overpass)...')
    buildings = ox.features_from_place(place, tags={'building': True})
    buildings = buildings[buildings.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    buildings = buildings.to_crs(crs)
    buildings['geometry'] = _repair_geometries(buildings.geometry)
    buildings = buildings[buildings.geometry.notna() & ~buildings.geometry.is_empty].copy()
    print(f'  âœ“ OSM: {len(buildings):,} building footprints')
    return buildings


def _load_from_pbf(place, crs, pbf_path):
    if not pbf_path or not Path(pbf_path).exists():
        raise FileNotFoundError('Local PBF file not found')

    print('  Trying local PBF...')
    from scripts.building_loader import get_borough_buildings

    buildings = get_borough_buildings(place, pbf_path=pbf_path, allow_overpass_fallback=False)
    if buildings is None or buildings.empty:
        raise RuntimeError('No building footprints returned from PBF')

    buildings = buildings.to_crs(crs)
    buildings['geometry'] = _repair_geometries(buildings.geometry)
    buildings = buildings[buildings.geometry.notna() & ~buildings.geometry.is_empty].copy()
    print(f'  âœ“ PBF: {len(buildings):,} building footprints')
    return buildings


def fetch_buildings(place, crs, pbf_path=None):
    """PBF-first for UK/England places; OSM-first elsewhere."""
    pbf_first = _is_uk_or_england_query(place)

    if pbf_first:
        try:
            print('  UK/England place detected â€” prioritizing local PBF...')
            return _load_from_pbf(place, crs, pbf_path)
        except Exception as e:
            print(f'  PBF failed: {e}')
            print('  Falling back to OSM (Overpass)...')
            try:
                return _load_from_osm(place, crs)
            except Exception as e2:
                print(f'  OSM failed: {e2}')
                print('  âš  No building footprints available â€” enclosure/visibility/land-use will be NaN')
                return None

    try:
        return _load_from_osm(place, crs)
    except Exception as e:
        print(f'  OSM failed: {e}')
        if pbf_path and Path(pbf_path).exists():
            try:
                print('  Falling back to local PBF...')
                return _load_from_pbf(place, crs, pbf_path)
            except Exception as e2:
                print(f'  PBF also failed: {e2}')
        print('  âš  No building footprints available â€” enclosure/visibility/land-use will be NaN')
        return None


buildings = fetch_buildings(FEATURE_QUERY, CRS_METRIC, PBF_PATH)

## Step 5: Compute Features

### 5.1 Segment Buffers

In [ ]:
segs_buf = edges[['geometry']].copy()
segs_buf['geometry'] = edges.geometry.buffer(BUFFER_M, cap_style=2)
print(f'âœ“ Segment buffers created ({BUFFER_M}m, flat ends)')

### 5.2 Lighting - Street Lamp Density

In [ ]:
MAPILLARY_DIR = Path(BASE_DIR) / 'notebooks' / 'data' / 'mapillary'
MAPILLARY_LIGHTS_FILES = {
    'raval, barcelona, spain': 'mapillary_lights_raval_barcelona.csv',
    'eixample, barcelona, spain': 'mapillary_lights_eixample_barcelona.csv',
    'belleville, paris, france': 'mapillary_lights_belleville_paris.csv',
    'trastevere, rome, italy': 'mapillary_lights_trastevere_rome.csv',
    'salamanca, madrid, spain': 'mapillary_lights_salamanca_madrid.csv',
    'london borough of islington, london, uk': 'mapillary_lights_islington_london.csv',
    'islington, london, uk': 'mapillary_lights_islington_london.csv',
}


def _load_mapillary_lights(place: str) -> gpd.GeoDataFrame | None:
    csv_name = MAPILLARY_LIGHTS_FILES.get(place.strip().lower())
    if csv_name is None:
        print(f'  âš  No Mapillary CSV mapping found for {place!r} â€” lighting set to 0')
        return None

    csv_path = MAPILLARY_DIR / csv_name
    if not csv_path.exists():
        print(f'  âš  Mapillary CSV not found: {csv_path} â€” lighting set to 0')
        return None

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.lower().str.strip()

    lon_col = next((c for c in df.columns if c.startswith('lon')), None)
    lat_col = next((c for c in df.columns if c.startswith('lat')), None)
    if lon_col is None or lat_col is None:
        raise ValueError(f'Could not find lon/lat columns in {csv_path}. Found: {list(df.columns)}')

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326',
    ).to_crs(CRS_METRIC)

    print(f'  âœ“ Loaded {len(gdf):,} Mapillary lights from {csv_path.name}')
    return gdf


try:
    print('Loading Mapillary street lights...')
    lamps_gdf = _load_mapillary_lights(FEATURE_QUERY)
    if lamps_gdf is None or lamps_gdf.empty:
        edges['lighting'] = 0
    else:
        lamp_counts = []
        for buf_geom in segs_buf.geometry:
            count = lamps_gdf.geometry.within(buf_geom).sum()
            lamp_counts.append(int(count))
        edges['lighting'] = lamp_counts
except Exception as e:
    print(f'  âš  Mapillary lamp load failed: {e} â€” lighting set to 0')
    edges['lighting'] = 0

print(f'  Lighting range: {edges["lighting"].min()} â€“ {edges["lighting"].max()}')

### 5.3 Connectivity â€” Node Degrees

In [ ]:
degree_map = dict(G.degree())
edges['connectivity'] = edges.apply(
    lambda row: degree_map.get(row['u'], 0) + degree_map.get(row['v'], 0),
    axis=1
)
print(f'âœ“ Connectivity range: {edges["connectivity"].min()} â€“ {edges["connectivity"].max()}')

### 5.4 Enclosure â€” Building Footprint Ã— Street Canyon

In [ ]:
def _segment_centroids(gdf):
    return np.column_stack((gdf.geometry.centroid.x.to_numpy(), gdf.geometry.centroid.y.to_numpy()))


def _fill_with_nearest_nonzero(edges_df, values, feature_name):
    series = pd.Series(values, index=edges_df.index, dtype='float64')
    missing = series.isna() | (series == 0)
    if not missing.any():
        return series

    valid = ~missing
    if not valid.any():
        print(f'  âš  {feature_name}: no non-zero values found â€” leaving zeros/NaNs as 0')
        return series.fillna(0.0)

    coords = _segment_centroids(edges_df)
    tree = cKDTree(coords[valid.to_numpy()])
    _, nearest = tree.query(coords[missing.to_numpy()], k=1)
    valid_positions = np.flatnonzero(valid.to_numpy())
    missing_positions = np.flatnonzero(missing.to_numpy())
    series.iloc[missing_positions] = series.iloc[valid_positions[nearest]].to_numpy()
    return series.fillna(0.0)


def compute_enclosure(edges, segs_buf, buildings):
    if buildings is None:
        print('  âš  No buildings â€” enclosure set to NaN')
        return np.full(len(edges), np.nan)

    # Building height: prefer 'height', fall back to levels Ã— 3m
    bld = buildings.copy()
    if 'height' in bld.columns:
        bld['height_m'] = pd.to_numeric(bld['height'], errors='coerce')
    else:
        bld['height_m'] = np.nan
    if 'building:levels' in bld.columns:
        levels = pd.to_numeric(bld['building:levels'], errors='coerce')
        bld['height_m'] = bld['height_m'].fillna(levels * 3.0)
    bld['height_m'] = bld['height_m'].fillna(9.0)  # 3-storey default

    # Street width from OSM tag, median fallback
    street_width = pd.to_numeric(edges.get('width', pd.Series(dtype=float)), errors='coerce')
    width_median = street_width.median()
    if pd.isna(width_median):
        width_median = 6.0

    buf_areas = segs_buf.geometry.area.values
    enclosure = np.zeros(len(segs_buf))

    enc_join = gpd.sjoin(bld[['geometry', 'height_m']], segs_buf[['geometry']],
                         how='inner', predicate='intersects')

    for seg_i, grp in enc_join.groupby('index_right'):
        buf_geom        = segs_buf.loc[seg_i, 'geometry']
        clipped_area    = bld.loc[grp.index, 'geometry'].intersection(buf_geom).area.sum()
        footprint_ratio = clipped_area / buf_areas[seg_i]
        h_mean          = bld.loc[grp.index, 'height_m'].mean()
        w_street        = street_width.iloc[seg_i] if seg_i < len(street_width) else width_median
        if pd.isna(w_street):
            w_street = width_median
        canyon_ratio      = h_mean / max(w_street, 1.0)
        canyon_normalized = min(canyon_ratio / 2.0, 1.0)
        enclosure[seg_i] = footprint_ratio * canyon_normalized

    return _fill_with_nearest_nonzero(edges, enclosure, 'enclosure')


edges['enclosure'] = compute_enclosure(edges, segs_buf, buildings)
print(f'âœ“ Enclosure range: {np.nanmin(edges["enclosure"]):.4f} â€“ {np.nanmax(edges["enclosure"]):.4f}')

### 5.5 Visibility â€” Building Blockage

In [ ]:
def compute_visibility(edges, buildings, buffer_m=BUFFER_VIS):
    if buildings is None:
        print('  âš  No buildings â€” visibility set to NaN')
        return np.full(len(edges), np.nan)

    vis_scores = []
    for geom in edges.geometry:
        buf = geom.buffer(buffer_m, cap_style=2)
        candidates = buildings[buildings.geometry.intersects(buf)]
        if candidates.empty:
            vis_scores.append(1.0)
            continue
        try:
            covered = candidates.geometry.intersection(buf).unary_union.area
        except Exception:
            covered = 0.0
        blocked = float(np.clip(covered / buf.area, 0.0, 1.0))
        vis_scores.append(round(1.0 - blocked, 4))
    return _fill_with_nearest_nonzero(edges, vis_scores, 'visibility')


edges['visibility'] = compute_visibility(edges, buildings)
print(f'âœ“ Visibility range: {np.nanmin(edges["visibility"]):.4f} â€“ {np.nanmax(edges["visibility"]):.4f}')

### 5.6 Dominant Land Use

In [ ]:
COMMERCIAL_BUILDING_VALUES = {
    'commercial', 'retail', 'industrial', 'office', 'warehouse',
    'supermarket', 'mall', 'hotel', 'school', 'hospital', 'train_station'
}
RESIDENTIAL_BUILDING_VALUES = {
    'house', 'residential', 'apartments', 'detached', 'semi_detached',
    'terrace', 'dormitory', 'bungalow', 'cabin', 'static_caravan'
}

def _classify_building_row(row):
    val = str(row.get('building', '')).strip().lower()
    if val in COMMERCIAL_BUILDING_VALUES: return 1.0
    if val in RESIDENTIAL_BUILDING_VALUES: return 0.0
    return 0.5

def compute_land_use(edges, buildings, buffer_m=BUFFER_VIS):
    if buildings is None:
        print('  âš  No buildings â€” land use set to NaN')
        return np.full(len(edges), np.nan)

    bld = buildings.copy()
    bld['building_use_score'] = bld.apply(_classify_building_row, axis=1)

    scores = []
    for geom in edges.geometry:
        buf        = geom.buffer(buffer_m, cap_style=2)
        candidates = bld[bld.geometry.intersects(buf)]
        if candidates.empty:
            scores.append(np.nan)
            continue
        try:
            intersections = candidates.geometry.intersection(buf)
        except Exception:
            scores.append(np.nan)
            continue
        areas      = intersections.area
        total_area = areas.sum()
        if total_area == 0:
            scores.append(np.nan)
            continue
        commercial  = (areas * candidates['building_use_score'].values).sum()
        residential = (areas * (1.0 - candidates['building_use_score'].values)).sum()
        raw    = (commercial - residential) / total_area
        mapped = round(float((raw + 1.0) / 2.0), 4)
        scores.append(mapped)
    return _fill_with_nearest_nonzero(edges, scores, 'land use')


edges['dominant_land_use_score'] = compute_land_use(edges, buildings)
print(f'âœ“ Land use range: {np.nanmin(edges["dominant_land_use_score"]):.4f} â€“ {np.nanmax(edges["dominant_land_use_score"]):.4f}')

### 5.7 Transport Proximity

In [ ]:
RAIL_TAGS = {
    'railway': ['subway_entrance', 'station', 'tram_stop'],
    'public_transport': 'stop_position',
}
BUS_TAGS = {
    'highway': 'bus_stop',
    'public_transport': 'platform',
}

def _nearest_dist(edges_proj, stops_gdf, col_name):
    segment_points = edges_proj.geometry.apply(
        lambda g: g.interpolate(0.5, normalized=True)
        if g.geom_type in ('LineString', 'MultiLineString') else g.centroid
    )
    if stops_gdf is None or stops_gdf.empty:
        edges_proj[col_name] = np.nan
        return edges_proj
    stop_coords = np.array([(g.x, g.y) for g in stops_gdf.geometry])
    dists = []
    for mp in segment_points:
        d = np.sqrt(((stop_coords - [mp.x, mp.y]) ** 2).sum(axis=1)).min()
        dists.append(round(float(d), 2))
    edges_proj[col_name] = dists
    return edges_proj

def fetch_stops(tags, label, place, crs):
    try:
        gdf = ox.features_from_place(place, tags=tags)
        gdf['geometry'] = gdf.geometry.apply(
            lambda g: g.centroid if g.geom_type != 'Point' else g
        )
        gdf = gdf[['geometry']].to_crs(crs).drop_duplicates('geometry')
        print(f'  {label}: {len(gdf):,} stops')
        return gdf
    except Exception as e:
        print(f'  âš  {label} fetch failed: {e}')
        return None

print('Fetching transit stops...')
rail = fetch_stops(RAIL_TAGS, 'Rail', FEATURE_QUERY, CRS_METRIC)
bus  = fetch_stops(BUS_TAGS,  'Bus',  FEATURE_QUERY, CRS_METRIC)

edges = _nearest_dist(edges, rail, 'dist_to_rail_m')
edges = _nearest_dist(edges, bus,  'dist_to_bus_m')

# Combined: use the closer of the two stops as the proximity score
edges['public_transport_proximity_m'] = edges[['dist_to_rail_m', 'dist_to_bus_m']].min(axis=1)
print(f'âœ“ Transport proximity range: {edges["public_transport_proximity_m"].min():.1f} â€“ {edges["public_transport_proximity_m"].max():.1f} m')

## Step 6: Normalize Features

Min-max normalization per feature, matching the method used on the London training data.

In [ ]:
def norm_col(series, clip=None):
    s = series.astype(float).copy()
    if clip is not None:
        s = s.clip(upper=clip)
    if s.isna().all():
        return pd.Series(0.0, index=s.index)
    s = s.fillna(s.median())
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / (mx - mn)

edges['lighting_norm'] = norm_col(edges.get('lighting', pd.Series(0.0, index=edges.index)))
edges['visibility_norm'] = norm_col(edges.get('visibility', pd.Series(0.0, index=edges.index)))
edges['connectivity_norm'] = norm_col(edges.get('connectivity', pd.Series(0.0, index=edges.index)))
edges['enclosure_norm'] = norm_col(edges.get('enclosure', pd.Series(0.0, index=edges.index)))
edges['dominant_land_use_score_norm'] = norm_col(edges.get('dominant_land_use_score', pd.Series(0.0, index=edges.index)))
edges['public_transport_proximity_m_norm'] = norm_col(edges.get('public_transport_proximity_m', pd.Series(0.0, index=edges.index)))

# Pedestrian priority score â€” normalize if available, otherwise default to zeros
if 'pedestrian_priority_score' in edges.columns:
    edges['pedestrian_priority_score_norm'] = norm_col(edges['pedestrian_priority_score'])
else:
    edges['pedestrian_priority_score_norm'] = pd.Series(0.0, index=edges.index)

FEATURE_COLS = [
    'lighting_norm', 'visibility_norm', 'connectivity_norm',
    'enclosure_norm', 'dominant_land_use_score_norm', 'public_transport_proximity_m_norm',
    'pedestrian_priority_score_norm'
]

print('âœ“ Features normalized')
print(edges[FEATURE_COLS].describe().round(4))

## Step 7: Classify Segments

In [ ]:
if 'scaler' not in globals() or 'model' not in globals():
    scaler = joblib.load(os.path.join(MODEL_DIR, 'scaler.pkl'))
    model  = joblib.load(os.path.join(MODEL_DIR, 'logistic_regression.pkl'))
    print('Loaded scaler/model in this cell (kernel state recovery).')

X = edges[FEATURE_COLS].values
X_scaled = scaler.transform(X)

edges['risk_class']       = model.predict(X_scaled)
proba                     = model.predict_proba(X_scaled)
edges['risk_probability'] = proba.max(axis=1).round(4)

print('âœ“ Classification complete')
print(f'\nRisk class distribution for: {PLACE}')
for cls in RISK_ORDER:
    n   = (edges['risk_class'] == cls).sum()
    pct = n / len(edges) * 100
    bar = 'â–ˆ' * int(pct / 2)
    print(f'  {cls:6s}: {n:5,}  ({pct:5.1f}%)  {bar}')

## Step 8: Export CSV

In [ ]:
place_slug = PLACE.replace(',', '').replace(' ', '_').lower()
output_folder = os.path.join(BASE_DIR, 'output', place_slug)
os.makedirs(output_folder, exist_ok=True)
csv_out = os.path.join(output_folder, f'{place_slug}_classified.csv')

raw_features   = ['lighting', 'visibility', 'connectivity', 'enclosure',
                   'dominant_land_use_score', 'public_transport_proximity_m',
                   'dist_to_rail_m', 'dist_to_bus_m', 'pedestrian_priority_score']
export_cols    = (['osmid'] if 'osmid' in edges.columns else []) + \
                 raw_features + FEATURE_COLS + ['risk_class', 'risk_probability']
export_cols    = [c for c in export_cols if c in edges.columns]

edges[export_cols].to_csv(csv_out, index=False)
print(f'âœ“ Exported to: {csv_out}')
print(f'  Columns: {export_cols}')

## Step 9: Map â€” Street Segments Coloured by Risk

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError(
        "plotly is required for this notebook view. Install it with: pip install plotly"
    ) from exc

from IPython.display import HTML, display

# Keep color mapping robust
edges['color'] = edges['risk_class'].map(RISK_COLORS).fillna('#7f8c8d')

# Segment details shown on hover (interactive in notebook)
detail_cols = [
    'risk_class', 'risk_probability', 'lighting', 'visibility', 'connectivity',
    'enclosure', 'dominant_land_use_score', 'public_transport_proximity_m',
    'pedestrian_priority_score'
]
# Also include normalized pedestrian priority if available
if 'pedestrian_priority_score_norm' in edges.columns:
    detail_cols.append('pedestrian_priority_score_norm')

if 'osmid' in edges.columns:
    detail_cols = ['osmid'] + detail_cols
detail_cols = [c for c in detail_cols if c in edges.columns]


def _to_lines(geom):
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == 'LineString':
        return [geom]
    if geom.geom_type == 'MultiLineString':
        return list(geom.geoms)
    return []


def _clean_value(v):
    if isinstance(v, (list, tuple, set, np.ndarray)):
        return ', '.join(str(x) for x in v)
    if v is None:
        return None
    if isinstance(v, (float, np.floating)):
        if np.isnan(v):
            return None
        return round(float(v), 4)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if pd.isna(v):
        return None
    return str(v)


fig = go.Figure()

for risk in RISK_ORDER:
    subset = edges[edges['risk_class'] == risk].copy()
    if subset.empty:
        continue

    xs, ys, custom = [], [], []

    for _, row in subset.iterrows():
        row_custom = [_clean_value(row.get(c, None)) for c in detail_cols]
        for line in _to_lines(row.geometry):
            lx, ly = line.xy
            n = len(lx)
            xs.extend(lx)
            ys.extend(ly)
            custom.extend([row_custom] * n)

            # Separator between lines
            xs.append(None)
            ys.append(None)
            custom.append([None] * len(detail_cols))

    hover_lines = [f"<b>{c.replace('_', ' ').title()}</b>: %{{customdata[{i}]}}" for i, c in enumerate(detail_cols)]
    hovertemplate = '<br>'.join(hover_lines) + '<extra></extra>'

    fig.add_trace(
        go.Scattergl(
            x=xs,
            y=ys,
            mode='lines',
            name=f"{risk.capitalize()} risk ({len(subset):,})",
            line=dict(color=RISK_COLORS[risk], width=2.2),
            customdata=custom,
            hovertemplate=hovertemplate,
        )
    )

fig.update_layout(
    title=f"Street Segment Safety Classification - {PLACE}",
    template='plotly_white',
    width=1200,
    height=760,
    legend_title_text='Risk Class',
    hovermode='closest',
    margin=dict(l=20, r=20, t=60, b=20),
)

# Hide axis ticks for cleaner map-like appearance
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, scaleanchor='x', scaleratio=1)

# Save into a place-specific folder under output/
map_folder = os.path.join(BASE_DIR, 'output', place_slug)
os.makedirs(map_folder, exist_ok=True)
map_out = os.path.join(map_folder, f'{place_slug}_risk_plotly.html')
map_png = os.path.join(map_folder, f'{place_slug}_risk_plotly.png')
fig.write_html(map_out, include_plotlyjs='cdn')

png_saved = False
try:
    fig.write_image(map_png)
    png_saved = True
except Exception as exc:
    print(f'  âš  PNG export skipped: {exc}')

print(f'âœ“ Interactive plot saved to: {map_out}')
if png_saved:
    print(f'âœ“ Static plot saved to: {map_png}')
print('  Zoom with wheel/trackpad, pan by dragging, hover segments to inspect features.')

# Render directly as HTML to avoid MIME renderer dependency issues in some kernels
display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False, config={'scrollZoom': True, 'displaylogo': False})))

## Step 10: Summary

In [ ]:
print('='*60)
print(f'CLASSIFICATION SUMMARY: {PLACE}')
print('='*60)
print(f'  Total segments classified : {len(edges):,}')
for cls in RISK_ORDER:
    n = (edges['risk_class'] == cls).sum()
    print(f'  {cls:6s} risk              : {n:6,} ({n/len(edges):.1%})')
print(f'  Mean prediction confidence: {edges["risk_probability"].mean():.3f}')
print(f'  CSV output : {csv_out}')
print(f'  Map output : {map_out}')
print()
print('Note: model was trained on London data. Classification of cities')
print('with significantly different spatial characteristics may be less reliable.')